# Restricted-family glass application

Model selection, identity prediction, calibrated intervals, and direct exceedance screening for the ten-oxide family. The default reruns every candidate rather than loading checkpoints. Execute a partir da raiz `code/fsnm`. O notebook grava figuras apenas em `code/fsnm/figures`; a cópia para o diretório TeX é deliberadamente manual.

## Configuration

In [1]:
import json
from itertools import product
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from sklearn.isotonic import IsotonicRegression

from fsnm import fit_fsnm


TARGET = "refractive_index"
SELECTED_OXIDES = [
    "tio2",
    "nb2o5",
    "ta2o5",
    "la2o3",
    "sio2",
    "b2o3",
    "p2o5",
    "k2o",
    "na2o",
    "li2o",
]
SPLIT_SEED = 2026
VALIDATION_FRACTION = 0.15
TEST_FRACTION = 0.15
ZERO_TOLERANCE = 1e-10
RANK_CANDIDATES = (5, 10, 20)
DEPTH_CANDIDATES = (3, 5)
LEAF_SIZE_CANDIDATES = (25, 50, 100)
MAX_ITERATIONS = 80
PATIENCE = 10
STEP_SIZE = 0.1
N_JOBS = 4
THRESHOLD = 1.8
QUANTILE_LEVELS = np.array([0.05, 0.15, 0.25, 0.50, 0.75, 0.85, 0.95])
BATCH_SIZE = 128

REUSE_CHECKPOINTS = False


## Filtering, splitting, and model selection

In [2]:
def load_selected_oxide_data(path):
    data = pd.read_parquet(path)
    composition_columns = [column for column in data if column != TARGET]
    missing = sorted(set(SELECTED_OXIDES) - set(composition_columns))
    if missing:
        raise ValueError(f"Missing selected oxide columns: {missing}")

    plausible_response = data[TARGET].between(1, 4.5)
    other_oxides = [
        column for column in composition_columns
        if column not in SELECTED_OXIDES
    ]
    other_mass = data[other_oxides].fillna(0).abs().sum(axis=1)
    selected_mass = data[SELECTED_OXIDES].fillna(0).abs().sum(axis=1)
    retained = (
        plausible_response
        & (other_mass < ZERO_TOLERANCE)
        & (selected_mass > ZERO_TOLERANCE)
    )
    restricted = data.loc[retained].copy()
    composition = restricted[SELECTED_OXIDES].fillna(0).to_numpy(
        float, copy=True,
    )
    composition /= composition.sum(axis=1, keepdims=True)
    response = restricted[TARGET].to_numpy(float)
    presence = (composition > ZERO_TOLERANCE).sum(axis=0)
    return composition, response, presence


def split_train_validation_test(composition, response):
    rng = np.random.default_rng(SPLIT_SEED)
    order = rng.permutation(len(response))
    n_validation = int(np.ceil(VALIDATION_FRACTION * len(order)))
    n_test = int(np.ceil(TEST_FRACTION * len(order)))
    validation_indices = order[:n_validation]
    test_indices = order[n_validation:n_validation + n_test]
    train_indices = order[n_validation + n_test:]
    return {
        "train": (composition[train_indices], response[train_indices]),
        "validation": (
            composition[validation_indices], response[validation_indices],
        ),
        "test": (composition[test_indices], response[test_indices]),
    }


def checkpoint_name(rank, max_depth, min_samples_leaf):
    return f"rank_{rank}_depth_{max_depth}_leaf_{min_samples_leaf}.joblib"


def fit_candidate(
    rank,
    max_depth,
    min_samples_leaf,
    x_train,
    y_train,
    x_validation,
    y_validation,
    checkpoint_directory,
):
    checkpoint_path = checkpoint_directory / checkpoint_name(
        rank, max_depth, min_samples_leaf,
    )
    if REUSE_CHECKPOINTS and checkpoint_path.exists():
        print(f"loading {checkpoint_path.name}", flush=True)
        return joblib.load(checkpoint_path)

    parameters = {
        "rank": rank,
        "n_iterations": MAX_ITERATIONS,
        "step_size": STEP_SIZE,
        "max_depth": max_depth,
        "min_samples_leaf": min_samples_leaf,
        "seed": 0,
        "patience": PATIENCE,
    }
    try:
        model = fit_fsnm(
            x_train,
            y_train,
            validation_data=(x_validation, y_validation),
            **parameters,
        )
    except np.linalg.LinAlgError as error:
        print(
            f"failed rank={rank}, depth={max_depth}, "
            f"leaf={min_samples_leaf}: {error}",
            flush=True,
        )
        return None

    history = model[3]
    best_iteration = history["best_iteration"]
    best_validation_loss = float(
        history["validation_loss"][best_iteration - 1]
    )
    candidate = {
        "rank": rank,
        "max_depth": max_depth,
        "min_samples_leaf": min_samples_leaf,
        "best_iteration": best_iteration,
        "best_validation_loss": best_validation_loss,
        "model": model,
    }
    temporary_path = checkpoint_path.with_suffix(".joblib.tmp")
    joblib.dump(candidate, temporary_path, compress=3)
    temporary_path.replace(checkpoint_path)
    print(
        f"finished rank={rank}, depth={max_depth}, leaf={min_samples_leaf}: "
        f"iteration={best_iteration}, loss={best_validation_loss:.6f}",
        flush=True,
    )
    return candidate


def select_model(parts, checkpoint_directory):
    checkpoint_directory.mkdir(parents=True, exist_ok=True)
    x_train, y_train = parts["train"]
    x_validation, y_validation = parts["validation"]
    configurations = list(product(
        RANK_CANDIDATES,
        DEPTH_CANDIDATES,
        LEAF_SIZE_CANDIDATES,
    ))
    candidates = Parallel(n_jobs=N_JOBS, verbose=10)(
        delayed(fit_candidate)(
            rank,
            depth,
            leaf,
            x_train,
            y_train,
            x_validation,
            y_validation,
            checkpoint_directory,
        )
        for rank, depth, leaf in configurations
    )
    candidates = [candidate for candidate in candidates if candidate is not None]
    if not candidates:
        raise RuntimeError("All hyperparameter candidates failed.")
    selected = min(
        candidates, key=lambda candidate: candidate["best_validation_loss"],
    )
    return selected, candidates


## Direct operator queries and validation calibration

In [3]:
def project_to_cdf(raw_cdf):
    """Project signed cumulative estimates onto monotone CDFs."""
    grid = np.arange(raw_cdf.shape[1] + 2)
    anchor_weight = 1e6
    projection_weights = np.ones(raw_cdf.shape[1] + 2)
    projection_weights[[0, -1]] = anchor_weight
    projected = np.empty_like(raw_cdf)
    projector = IsotonicRegression(y_min=0.0, y_max=1.0, increasing=True)
    for row in range(len(raw_cdf)):
        augmented = np.concatenate(([0.0], raw_cdf[row], [1.0]))
        projected[row] = projector.fit_transform(
            grid, augmented, sample_weight=projection_weights,
        )[1:-1]
    return projected


def conditional_diagnostics(
    model,
    x,
    y_observed,
    y_marginal,
    threshold=THRESHOLD,
    quantile_levels=QUANTILE_LEVELS,
):
    phi_model, psi_model, singular_values, _ = model
    psi_marginal = psi_model.predict(y_marginal[:, None])
    marginal_order = np.argsort(y_marginal)
    y_sorted = y_marginal[marginal_order]

    quantiles = np.empty((len(x), len(quantile_levels)))
    exceedance_probability = np.empty(len(x))
    conditional_mean = np.empty(len(x))
    pit = np.empty(len(x))
    negative_fraction = np.empty(len(x))
    raw_mass = np.empty(len(x))
    clipped_mass = np.empty(len(x))
    effective_sample_size = np.empty(len(x))
    crps = np.empty(len(x))

    for start in range(0, len(x), BATCH_SIZE):
        end = min(start + BATCH_SIZE, len(x))
        phi = phi_model.predict(x[start:end])
        raw_weights = 1 + (phi * singular_values) @ psi_marginal.T
        sorted_weights = raw_weights[:, marginal_order]
        raw_cdf = np.cumsum(sorted_weights, axis=1) / len(y_marginal)
        projected_cdf = project_to_cdf(raw_cdf)
        for index, level in enumerate(quantile_levels):
            locations = (projected_cdf >= level).argmax(axis=1)
            quantiles[start:end, index] = y_sorted[locations]

        exceedance_probability[start:end] = np.clip(
            np.mean(
                raw_weights
                * (y_marginal[None, :] > threshold),
                axis=1,
            ),
            0.0,
            1.0,
        )
        conditional_mean[start:end] = np.mean(
            raw_weights * y_marginal[None, :], axis=1,
        )
        observed_locations = np.searchsorted(
            y_sorted, y_observed[start:end], side="right",
        ) - 1
        batch_pit = np.zeros(end - start)
        has_marginal_value_below = observed_locations >= 0
        batch_rows = np.arange(end - start)[has_marginal_value_below]
        batch_pit[has_marginal_value_below] = projected_cdf[
            batch_rows, observed_locations[has_marginal_value_below]
        ]
        pit[start:end] = batch_pit
        negative_fraction[start:end] = (raw_weights < 0).mean(axis=1)
        raw_mass[start:end] = raw_weights.mean(axis=1)
        clipped_mass[start:end] = np.maximum(raw_weights, 0.0).mean(axis=1)
        effective_sample_size[start:end] = (
            raw_weights.sum(axis=1) ** 2 / np.sum(raw_weights**2, axis=1)
        )
        interval_widths = np.diff(y_sorted)
        observed_step = (
            y_observed[start:end, None] <= y_sorted[None, :-1]
        )
        crps[start:end] = np.sum(
            (projected_cdf[:, :-1] - observed_step) ** 2
            * interval_widths[None, :],
            axis=1,
        )

    return {
        "quantiles": quantiles,
        "exceedance_probability": exceedance_probability,
        "conditional_mean": conditional_mean,
        "pit": pit,
        "negative_fraction": negative_fraction,
        "raw_mass": raw_mass,
        "clipped_mass": clipped_mass,
        "effective_sample_size": effective_sample_size,
        "crps": crps,
    }


def regression_metrics(observed, predicted):
    errors = predicted - observed
    mse = float(np.mean(errors**2))
    if np.std(predicted) > 0 and np.std(observed) > 0:
        correlation = float(np.corrcoef(observed, predicted)[0, 1])
    else:
        correlation = None
    return {
        "mse": mse,
        "rmse": float(np.sqrt(mse)),
        "mae": float(np.mean(np.abs(errors))),
        "r_squared": float(
            1 - np.sum(errors**2) / np.sum((observed - observed.mean())**2)
        ),
        "bias": float(errors.mean()),
        "correlation": correlation,
    }


def reliability_bins(observed, predicted, n_bins=8):
    order = np.argsort(predicted)
    bins = np.array_split(order, n_bins)
    predicted_means = np.array([predicted[index].mean() for index in bins])
    observed_means = np.array([observed[index].mean() for index in bins])
    errors = np.array([
        np.sqrt(observed_means[i] * (1 - observed_means[i]) / len(index))
        for i, index in enumerate(bins)
    ])
    return predicted_means, observed_means, errors


def interval_metrics(y_test, quantiles, y_train):
    level_to_index = {
        float(level): index for index, level in enumerate(QUANTILE_LEVELS)
    }
    intervals = [(0.50, 0.25, 0.75), (0.70, 0.15, 0.85), (0.90, 0.05, 0.95)]
    results = {}
    for nominal, lower_level, upper_level in intervals:
        lower = quantiles[:, level_to_index[lower_level]]
        upper = quantiles[:, level_to_index[upper_level]]
        unconditional_lower, unconditional_upper = np.quantile(
            y_train, [lower_level, upper_level],
        )
        results[f"{int(100 * nominal)}"] = {
            "nominal": nominal,
            "coverage": float(np.mean((y_test >= lower) & (y_test <= upper))),
            "mean_width": float(np.mean(upper - lower)),
            "unconditional_coverage": float(np.mean(
                (y_test >= unconditional_lower) & (y_test <= unconditional_upper)
            )),
            "unconditional_width": float(
                unconditional_upper - unconditional_lower
            ),
        }
    return results


def validation_uq_summary(candidate, parts):
    x_train, y_train = parts["train"]
    x_validation, y_validation = parts["validation"]
    diagnostics = conditional_diagnostics(
        candidate["model"], x_validation, y_validation, y_train,
    )
    intervals = interval_metrics(
        y_validation, diagnostics["quantiles"], y_train,
    )
    calibration_error = float(np.mean([
        abs(values["coverage"] - values["nominal"])
        for values in intervals.values()
    ]))
    observed_exceedance = (y_validation > THRESHOLD).astype(float)
    brier_score = float(np.mean(
        (diagnostics["exceedance_probability"] - observed_exceedance) ** 2
    ))
    return {
        "rank": candidate["rank"],
        "max_depth": candidate["max_depth"],
        "min_samples_leaf": candidate["min_samples_leaf"],
        "best_iteration": candidate["best_iteration"],
        "validation_loss": candidate["best_validation_loss"],
        "mean_crps": float(np.mean(diagnostics["crps"])),
        "interval_calibration_error": calibration_error,
        "conditional_mean_rmse": regression_metrics(
            y_validation, diagnostics["conditional_mean"],
        )["rmse"],
        "screening_brier_score": brier_score,
        "negative_weight_fraction": float(np.mean(
            diagnostics["negative_fraction"],
        )),
        "intervals": intervals,
    }


def test_summary(candidate, parts):
    _, y_train = parts["train"]
    x_test, y_test = parts["test"]
    diagnostics = conditional_diagnostics(
        candidate["model"], x_test, y_test, y_train,
    )
    intervals = interval_metrics(y_test, diagnostics["quantiles"], y_train)
    prediction = regression_metrics(y_test, diagnostics["conditional_mean"])
    observed_exceedance = (y_test > THRESHOLD).astype(float)
    brier_score = float(np.mean(
        (diagnostics["exceedance_probability"] - observed_exceedance) ** 2
    ))
    test_rate = float(observed_exceedance.mean())
    training_rate = float(np.mean(y_train > THRESHOLD))
    baseline_brier = float(np.mean((training_rate - observed_exceedance) ** 2))
    pit_sorted = np.sort(diagnostics["pit"])
    uniform_grid = (np.arange(len(pit_sorted)) + 0.5) / len(pit_sorted)
    pit_ks_distance = float(np.max(np.abs(pit_sorted - uniform_grid)))
    return diagnostics, intervals, {
        "conditional_mean_metrics": prediction,
        "mean_crps": float(np.mean(diagnostics["crps"])),
        "intervals": intervals,
        "screening": {
            "threshold": THRESHOLD,
            "training_exceedance_rate": training_rate,
            "test_exceedance_rate": test_rate,
            "brier_score": brier_score,
            "training_rate_brier_score": baseline_brier,
        },
        "uq_diagnostics": {
            "pit_ks_distance": pit_ks_distance,
            "negative_weight_fraction": distribution_summary(
                diagnostics["negative_fraction"],
            ),
            "raw_kernel_mass": distribution_summary(diagnostics["raw_mass"]),
            "clipped_kernel_mass": distribution_summary(
                diagnostics["clipped_mass"],
            ),
            "effective_marginal_sample_size": distribution_summary(
                diagnostics["effective_sample_size"],
            ),
        },
    }


def calibrated_test_summary(candidate, parts):
    x_train, y_train = parts["train"]
    x_validation, y_validation = parts["validation"]
    x_test, y_test = parts["test"]
    validation = conditional_diagnostics(
        candidate["model"], x_validation, y_validation, y_train,
    )
    calibrated_raw_levels = np.quantile(
        validation["pit"], QUANTILE_LEVELS,
    )
    test = conditional_diagnostics(
        candidate["model"],
        x_test,
        y_test,
        y_train,
        quantile_levels=calibrated_raw_levels,
    )
    intervals = interval_metrics(y_test, test["quantiles"], y_train)

    validation_binary = (y_validation > THRESHOLD).astype(float)
    test_binary = (y_test > THRESHOLD).astype(float)
    probability_calibrator = IsotonicRegression(
        y_min=0.0, y_max=1.0, out_of_bounds="clip",
    )
    probability_calibrator.fit(
        validation["exceedance_probability"], validation_binary,
    )
    calibrated_probability = probability_calibrator.predict(
        test["exceedance_probability"],
    )
    calibrated_brier = float(np.mean(
        (calibrated_probability - test_binary) ** 2
    ))

    sorted_validation_pit = np.sort(validation["pit"])
    recalibrated_test_pit = np.searchsorted(
        sorted_validation_pit, test["pit"], side="right",
    ) / len(sorted_validation_pit)
    sorted_test_pit = np.sort(recalibrated_test_pit)
    uniform_grid = (np.arange(len(sorted_test_pit)) + 0.5) / len(sorted_test_pit)
    return test, intervals, {
        "raw_levels_for_target_quantiles": dict(zip(
            map(str, QUANTILE_LEVELS), map(float, calibrated_raw_levels),
        )),
        "intervals": intervals,
        "recalibrated_pit_ks_distance": float(np.max(np.abs(
            sorted_test_pit - uniform_grid
        ))),
        "isotonic_screening_brier_score": calibrated_brier,
        "calibrated_screening_probability": distribution_summary(
            calibrated_probability,
        ),
    }


## Paper figures

In [4]:
def save_kernel_figure(selected, figure_path):
    history = selected["model"][3]
    singular_values = selected["model"][2]
    energy_shares = singular_values**2 / np.sum(singular_values**2)
    figure, axes = plt.subplots(1, 2, figsize=(10.8, 3.7), constrained_layout=True)

    iterations = np.arange(1, len(history["training_loss"]) + 1)
    axes[0].plot(
        iterations, history["training_loss"],
        color="tab:blue", linewidth=1.8, label="Training",
    )
    axes[0].plot(
        iterations, history["validation_loss"],
        color="tab:orange", linewidth=1.8, label="Validation",
    )
    axes[0].axvline(
        selected["best_iteration"], color="black", linestyle="--",
        linewidth=1.2, label=f"Selected: {selected['best_iteration']}",
    )
    axes[0].set(
        title=(
            f"Selection: rank {selected['rank']}, depth "
            f"{selected['max_depth']}"
        ),
        xlabel="Iteration", ylabel="Empirical FSNM loss",
    )
    axes[0].legend(frameon=False)

    locations = np.arange(1, len(singular_values) + 1)
    axes[1].bar(
        locations, singular_values, color="tab:blue", label="Singular value",
    )
    axes[1].set(
        title="Selected dependence spectrum", xlabel="Mode",
        ylabel="Singular value", xlim=(0.25, len(singular_values) + 0.75),
        ylim=(0, 1.08 * singular_values.max()),
    )
    energy_axis = axes[1].twinx()
    energy_axis.plot(
        locations, np.cumsum(energy_shares), color="tab:orange",
        marker="o", markersize=3, linewidth=1.3, label="Cumulative energy",
    )
    energy_axis.set(ylabel="Cumulative energy", ylim=(0, 1.04))
    handles_left, labels_left = axes[1].get_legend_handles_labels()
    handles_right, labels_right = energy_axis.get_legend_handles_labels()
    axes[1].legend(
        handles_left + handles_right, labels_left + labels_right,
        frameon=False, loc="center right",
    )
    figure.savefig(figure_path, dpi=220, bbox_inches="tight")
    plt.close(figure)


def calibration_bins(observed, predicted, n_bins=8):
    order = np.argsort(predicted)
    bins = np.array_split(order, n_bins)
    predicted_means = np.array([predicted[index].mean() for index in bins])
    observed_means = np.array([observed[index].mean() for index in bins])
    standard_errors = np.array([
        observed[index].std(ddof=1) / np.sqrt(len(index)) for index in bins
    ])
    return predicted_means, observed_means, standard_errors


def save_prediction_figure(y_test, predicted, training_mean, metrics, figure_path):
    figure, axes = plt.subplots(1, 2, figsize=(10.2, 4.1), constrained_layout=True)
    limits = (min(y_test.min(), predicted.min()), max(y_test.max(), predicted.max()))
    density = axes[0].hexbin(
        y_test, predicted, gridsize=32, mincnt=1, bins="log", cmap="viridis",
    )
    axes[0].plot(limits, limits, color="black", linestyle="--", linewidth=1.2)
    axes[0].axhline(
        training_mean, color="tab:red", linestyle=":", linewidth=1.3,
        label="Training mean",
    )
    axes[0].set(
        title="Held-out RI prediction", xlabel="Observed refractive index",
        ylabel="Predicted refractive index", xlim=limits, ylim=limits,
    )
    axes[0].legend(frameon=False)
    figure.colorbar(density, ax=axes[0], label="log count")

    predicted_means, observed_means, standard_errors = calibration_bins(
        y_test, predicted,
    )
    lower = min(predicted_means.min(), np.min(observed_means - 1.96 * standard_errors))
    upper = max(predicted_means.max(), np.max(observed_means + 1.96 * standard_errors))
    padding = 0.08 * (upper - lower)
    calibration_limits = (lower - padding, upper + padding)
    axes[1].errorbar(
        predicted_means, observed_means, yerr=1.96 * standard_errors,
        fmt="o-", color="tab:blue", capsize=3, label="Prediction octiles",
    )
    axes[1].plot(
        calibration_limits, calibration_limits,
        color="black", linestyle="--", linewidth=1.2,
    )
    axes[1].set(
        title=(
            f"RMSE={metrics['rmse']:.3f}, $R^2$={metrics['r_squared']:.3f}"
        ),
        xlabel="Mean predicted RI", ylabel="Mean observed RI",
        xlim=calibration_limits, ylim=calibration_limits,
    )
    axes[1].legend(frameon=False)
    figure.savefig(figure_path, dpi=220, bbox_inches="tight")
    plt.close(figure)


def save_conditional_figure(
    y_test,
    calibrated_quantiles,
    screening_probability,
    intervals,
    figure_path,
    seed=0,
):
    figure, axes = plt.subplots(1, 2, figsize=(11.5, 4.2), constrained_layout=True)
    level_to_index = {
        float(level): index for index, level in enumerate(QUANTILE_LEVELS)
    }
    rng = np.random.default_rng(seed)
    sample_size = min(150, len(y_test))
    sample = rng.choice(len(y_test), size=sample_size, replace=False)
    median = calibrated_quantiles[:, level_to_index[0.50]]
    sample = sample[np.argsort(median[sample])]
    x_axis = np.arange(sample_size)

    axes[0].fill_between(
        x_axis,
        calibrated_quantiles[sample, level_to_index[0.05]],
        calibrated_quantiles[sample, level_to_index[0.95]],
        color="tab:blue", alpha=0.18, label="90% interval",
    )
    axes[0].fill_between(
        x_axis,
        calibrated_quantiles[sample, level_to_index[0.15]],
        calibrated_quantiles[sample, level_to_index[0.85]],
        color="tab:blue", alpha=0.35, label="70% interval",
    )
    axes[0].plot(
        x_axis, median[sample], color="tab:blue", linewidth=1.3,
        label="Conditional median",
    )
    axes[0].scatter(
        x_axis, y_test[sample], color="black", s=9, zorder=5,
        label="Observed RI",
    )
    axes[0].set(
        title=(
            "Validation-calibrated intervals "
            f"(90% coverage={intervals['90']['coverage']:.3f})"
        ),
        xlabel="Test observation (sorted by median)",
        ylabel="Refractive index",
    )
    axes[0].legend(frameon=False, fontsize=8, loc="upper left")

    observed_exceedance = (y_test > THRESHOLD).astype(float)
    predicted_means, observed_means, errors = reliability_bins(
        observed_exceedance, screening_probability,
    )
    axes[1].errorbar(
        predicted_means, observed_means, yerr=1.96 * errors,
        fmt="o-", color="tab:red", capsize=3,
    )
    axes[1].plot([0, 1], [0, 1], color="black", linestyle="--", linewidth=1.2)
    axes[1].set(
        title=rf"Direct screening probability ($\mathrm{{RI}}>{THRESHOLD}$)",
        xlabel="Predicted exceedance probability",
        ylabel="Observed exceedance frequency",
        xlim=(-0.02, 1.02), ylim=(-0.02, 1.02),
    )
    figure.savefig(figure_path, dpi=220, bbox_inches="tight")
    plt.close(figure)


def distribution_summary(values):
    return {
        "mean": float(np.mean(values)),
        "standard_deviation": float(np.std(values)),
        "minimum": float(np.min(values)),
        "median": float(np.median(values)),
        "maximum": float(np.max(values)),
    }


## Run the complete experiment

In [5]:

package_directory = Path.cwd()
composition, response, presence = load_selected_oxide_data(
    package_directory / "data" / "refractive_index.parquet",
)
parts = split_train_validation_test(composition, response)
checkpoint_directory = package_directory / "artifacts" / "glass_selected_oxides_search"
loss_selected, candidates = select_model(parts, checkpoint_directory)

validation_summaries = [validation_uq_summary(candidate, parts) for candidate in candidates]
crps_summary = min(validation_summaries, key=lambda summary: summary["mean_crps"])
crps_selected = next(
    candidate for candidate in candidates
    if (candidate["rank"], candidate["max_depth"], candidate["min_samples_leaf"])
    == (crps_summary["rank"], crps_summary["max_depth"], crps_summary["min_samples_leaf"])
)

_, y_train = parts["train"]
_, y_test = parts["test"]
loss_diagnostics, loss_intervals, loss_test = test_summary(loss_selected, parts)
crps_diagnostics, crps_intervals, crps_test = test_summary(crps_selected, parts)
calibrated_diagnostics, calibrated_intervals, calibrated_test = calibrated_test_summary(
    crps_selected, parts,
)
baseline_prediction = regression_metrics(y_test, np.full_like(y_test, y_train.mean()))

figure_directory = package_directory / "figures"
figure_directory.mkdir(exist_ok=True)
figure_paths = {
    "kernel": figure_directory / "11_glass_kernel_fit.png",
    "prediction": figure_directory / "12_glass_identity_prediction.png",
    "conditional": figure_directory / "13_glass_conditional_queries.png",
}
save_kernel_figure(crps_selected, figure_paths["kernel"])
save_prediction_figure(
    y_test, crps_diagnostics["conditional_mean"], y_train.mean(),
    crps_test["conditional_mean_metrics"], figure_paths["prediction"],
)
save_conditional_figure(
    y_test, calibrated_diagnostics["quantiles"],
    crps_diagnostics["exceedance_probability"], calibrated_intervals,
    figure_paths["conditional"],
)

results = {
    "selected_oxides": SELECTED_OXIDES,
    "zero_tolerance_for_other_oxides": ZERO_TOLERANCE,
    "observations": len(response),
    "split_sizes": {name: len(values[1]) for name, values in parts.items()},
    "oxide_positive_counts": dict(zip(SELECTED_OXIDES, map(int, presence))),
    "response_summary": distribution_summary(response),
    "loss_selected_model": {
        "rank": loss_selected["rank"],
        "max_depth": loss_selected["max_depth"],
        "min_samples_leaf": loss_selected["min_samples_leaf"],
        "best_iteration": loss_selected["best_iteration"],
        "best_validation_loss": loss_selected["best_validation_loss"],
        "singular_values": loss_selected["model"][2].tolist(),
    },
    "crps_selected_model": {
        "rank": crps_selected["rank"],
        "max_depth": crps_selected["max_depth"],
        "min_samples_leaf": crps_selected["min_samples_leaf"],
        "best_iteration": crps_selected["best_iteration"],
        "validation_mean_crps": crps_summary["mean_crps"],
        "singular_values": crps_selected["model"][2].tolist(),
    },
    "candidate_results": [
        {
            "rank": candidate["rank"],
            "max_depth": candidate["max_depth"],
            "min_samples_leaf": candidate["min_samples_leaf"],
            "best_iteration": candidate["best_iteration"],
            "best_validation_loss": candidate["best_validation_loss"],
        }
        for candidate in sorted(candidates, key=lambda value: value["best_validation_loss"])
    ],
    "mean_baseline_metrics": baseline_prediction,
    "validation_uq_results": sorted(validation_summaries, key=lambda summary: summary["mean_crps"]),
    "loss_selected_test_results": loss_test,
    "crps_selected_test_results": crps_test,
    "validation_recalibrated_test_results": calibrated_test,
}
results_path = package_directory / "artifacts" / "glass_selected_oxides_uq.json"
results_path.write_text(json.dumps(results, indent=2) + "\n")

print("observations and split:", results["observations"], results["split_sizes"])
print("loss-selected model:", results["loss_selected_model"] | {"singular_values": "omitted"})
print("CRPS-selected model:", results["crps_selected_model"] | {"singular_values": "omitted"})
print("identity metrics:", crps_test["conditional_mean_metrics"])
print("calibrated intervals:", calibrated_intervals)
print("screening:", crps_test["screening"])
for path in figure_paths.values():
    print("saved:", path)
print("saved:", results_path)


[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.


finished rank=5, depth=3, leaf=100: iteration=11, loss=-1.695268


finished rank=5, depth=3, leaf=50: iteration=16, loss=-1.744680


finished rank=5, depth=3, leaf=25: iteration=25, loss=-2.239035


finished rank=5, depth=5, leaf=100: iteration=18, loss=-2.298724


...

finished rank=20, depth=5, leaf=100: iteration=15, loss=-3.593763


finished rank=20, depth=5, leaf=50: iteration=18, loss=-4.309734


finished rank=20, depth=5, leaf=25: iteration=39, loss=-5.647596


[Parallel(n_jobs=4)]: Done  18 out of  18 | elapsed:  1.3min finished


observations and split: 3013 {'train': 2109, 'validation': 452, 'test': 452}
loss-selected model: {'rank': 20, 'max_depth': 5, 'min_samples_leaf': 25, 'best_iteration': 39, 'best_validation_loss': -5.6475961556590555, 'singular_values': 'omitted'}
CRPS-selected model: {'rank': 20, 'max_depth': 5, 'min_samples_leaf': 25, 'best_iteration': 39, 'validation_mean_crps': 0.020270202946975638, 'singular_values': 'omitted'}
identity metrics: {'mse': 0.00505705472447288, 'rmse': 0.07111297156266837, 'mae': 0.034545338358776566, 'r_squared': 0.8109297778393831, 'bias': -0.0060231810910154684, 'correlation': 0.9015901172545846}
calibrated intervals: {'50': {'nominal': 0.5, 'coverage': 0.5464601769911505, 'mean_width': 0.043368252212389366, 'unconditional_coverage': 0.5132743362831859, 'unconditional_width': 0.133}, '70': {'nominal': 0.7, 'coverage': 0.7146017699115044, 'mean_width': 0.06942865044247787, 'unconditional_coverage': 0.6969026548672567, 'unconditional_width': 0.24261999999999984}, '90